## Student Attendance Analytics with Pandas

In [ ]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [ ]:
df = pd.read_excel("attendance.xlsx")
df

#### Section 1: Data Inspection

In [ ]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
#-----------------------------Write Solution Code-----------------------------
df.head(10)
df.shape
df.dtypes

In [ ]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
#-----------------------------Write Solution Code-----------------------------
df['Last Present Date'] = df['Last Present Date'].replace('Never', pd.NA)
df.isna().sum()

In [ ]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
#-----------------------------Write Solution Code-----------------------------
df.columns = (
    df.columns
      .str.lower()
      .str.replace('%', 'pct', regex=False)
      .str.replace(' ', '_', regex=False)
)

df.columns

In [ ]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
#-----------------------------Write Solution Code-----------------------------
df['attendance_pct'] = pd.to_numeric(df['attendance_pct'], errors='coerce') / 100

df['last_present_date'] = pd.to_datetime(
    df['last_present_date'].replace('Never', pd.NaT),
    errors='coerce'
)

#### Section 2: Filtering & Selection

In [ ]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
#-----------------------------Write Solution Code-----------------------------
df[df['status'] == 'Critical'].sort_values('attendance_pct')


In [ ]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
#-----------------------------Write Solution Code-----------------------------
df.loc[df['present'] == 0, ['roll_no', 'student_name', 'consecutive_absences']]

In [ ]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """
#-----------------------------Write Solution Code-----------------------------
df.loc[df['status'] == 'Safe']
df.iloc[:15, 2:7]

In [ ]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
#-----------------------------Write Solution Code-----------------------------
df.loc[
    df['consecutive_absences'] >= 5,
    ['student_name', 'status', 'last_present_date']
]

### Section 3: Grouping & Aggregation 

In [ ]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
#-----------------------------Write Solution Code-----------------------------
df.groupby('status').agg(
    student_count=('student_name', 'count'),
    avg_attendance_pct=('attendance_pct', 'mean'),
    max_consecutive_absences=('consecutive_absences', 'max')
)



In [ ]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""
#-----------------------------Write Solution Code-----------------------------
df['attendance_category'] = pd.cut(
    df['attendance_pct'],
    bins=[0, 60, 75, 90, 100],
    labels=['Poor', 'Average', 'Good', 'Excellent'],
    include_lowest=True
)

df.groupby('attendance_category').size()

In [ ]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
#-----------------------------Write Solution Code-----------------------------
df.agg(
    total_classes_conducted=('total_classes', 'sum'),
    total_present=('present', 'sum'),
    total_absent=('absent', 'sum')
)

In [ ]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
#-----------------------------Write Solution Code-----------------------------
df.groupby('status').agg(
    min_present=('present', 'min'),
    max_present=('present', 'max'),
    mean_present=('present', 'mean'),
    total_absent=('absent', 'sum')
)

### Section 4: Feature Engineering & Transformation

In [ ]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
#-----------------------------Write Solution Code-----------------------------
df['risk_score'] = (
    df['absent'] * 0.4 +
    df['consecutive_absences'] * 0.6
)

df.nlargest(10, 'risk_score')

In [34]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
#-----------------------------Write Solution Code-----------------------------
col = [c for c in df.columns if 'classes_needed' in c][0]

df['classes_needed_numeric'] = (
    df[col]
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .fillna(0)
    .astype(int)
)

df[['classes_needed_numeric']].head()


,classes_needed_numeric
0,33
1,5
2,0
3,0
4,0


In [37]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
#-----------------------------Write Solution Code-----------------------------
col = [c for c in df.columns if 'classes_needed' in c][0]

df['classes_needed_numeric'] = (
    df[col]
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .fillna(0)
    .astype(int)
)

df[[col, 'classes_needed_numeric']]

,classes_needed_(75pct),classes_needed_numeric
0,33 more,33
1,5 more,5
2,✓ Met,0
3,✓ Met,0
4,✓ Met,0
...,...,...
57,✓ Met,0
58,30 more,30
59,10 more,10
60,6 more,6


In [38]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
#-----------------------------Write Solution Code-----------------------------
df['attendance_range'] = pd.cut(
    df['attendance_pct'],
    bins=[0, 50, 75, 90, 100],
    labels=['Critical', 'Warning', 'Good', 'Excellent'],
    include_lowest=True
)

df[['attendance_pct', 'attendance_range']]

,attendance_pct,attendance_range
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
...,...,...
57,NaN,NaN
58,NaN,NaN
59,NaN,NaN
60,NaN,NaN


### Section 5: Advanced Analysis 

In [39]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
#-----------------------------Write Solution Code-----------------------------
pd.pivot_table(
    df,
    index='status',
    columns='attendance_category',
    values='student_name',
    aggfunc='count',
    fill_value=0
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_26592\1422218583.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pd.pivot_table(


attendance_category,Poor,Average,Good,Excellent
status,,,,
Critical,0,0,0,0
Safe,0,0,0,0
Warning,0,0,0,0


In [40]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
#-----------------------------Write Solution Code-----------------------------
df['base_name'] = df['student_name'].str.split().str[0]

df.loc[
    df['base_name'].duplicated(keep=False),
    ['roll_no', 'student_name', 'base_name']
].sort_values('base_name')


,roll_no,student_name,base_name
18,4MW24AD019,MOHAMMED SUHAN,MOHAMMED
19,4MW24AD020,MOHAMMED AFREED,MOHAMMED
20,4MW24AD021,MOHAMMED AWAD TONSE,MOHAMMED
24,4MW24AD025,PAVAN BHAT,PAVAN
25,4MW24AD026,PAVAN S POOJARY,PAVAN


In [41]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
#-----------------------------Write Solution Code-----------------------------
student_details = pd.DataFrame({
    'roll_no': ['4MW24AD001', '4MW24AD002', '4MW24AD003', '4MW24AD004'],
    'branch': ['AI&DS', 'AI&DS', 'AI&DS', 'AI&DS'],
    'semester': [5, 5, 5, 5]
})

merged_df = pd.merge(
    df,
    student_details,
    on='roll_no',
    how='inner'
)

merged_df

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_(75pct),last_present_date,consecutive_absences,attendance_category,risk_score,attendance_flag,classes_needed_numeric,attendance_range,base_name,branch,semester
0,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,NaN,Critical,5 more,2026-10-07,0,NaN,1.6,Safe,5,NaN,ADITHYA,AI&DS,5
1,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,NaN,Safe,✓ Met,2026-10-07,0,NaN,0.0,Safe,0,NaN,ANKITHA,AI&DS,5
2,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,NaN,Safe,✓ Met,2026-10-07,0,NaN,0.4,Safe,0,NaN,ANUSHA,AI&DS,5
3,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,NaN,Warning,✓ Met,2026-10-07,0,NaN,0.8,Safe,0,NaN,ANVITA,AI&DS,5


In [45]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
#-----------------------------Write Solution Code-----------------------------
critical_students = df[df['status'] == 'Critical']
critical_students.to_csv('critical_students.csv', index=False)

status_summary = df.groupby('status').agg(
    student_count=('student_name', 'count'),
    avg_attendance_pct=('attendance_pct', 'mean'),
    max_consecutive_absences=('consecutive_absences', 'max')
)

status_summary.to_csv('status_summary.csv')

top_10_risk = df.nlargest(10, 'risk_score')
top_10_risk.to_csv('top_10_risk_students.csv', index=False)